# JBot Database Health Check

This notebook analyzes the health and engagement metrics of the `jbot.db` database. It provides insights into database size, user engagement, question usage, and more.


## 1. Setup and Database Connection

This notebook assumes that `jbot.db` is in `./.database/`. If it's not, please update the `db_file` variable in the next cell to point to the correct location.

The following code will connect to the database.

In [ ]:
import sqlite3
import pandas as pd
import os
import plotly.express as px

db_file = 'jbot.db'

if not os.path.exists(db_file):
    print(f"Database file '{db_file}' not found. Please ensure it is in the same directory as this notebook.")
    conn = None
else:
    conn = sqlite3.connect(db_file)
    print("Successfully connected to the database.")

c:\Users\Blake\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Successfully connected to the database.


## 2. Database Size and Record Counts

This section provides a high-level overview of the database's size on disk and the number of records in each table.


In [2]:
if 'conn' in locals() and conn:
    # Get database file size
    db_size = os.path.getsize(db_file)
    print(f"Database file size: {db_size / 1024:.2f} KB")

    # Get record counts for each table
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    record_counts = []
    for table_name in tables:
        table_name = table_name[0]
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        record_counts.append({'Table': table_name, 'Record Count': count})

    df_counts = pd.DataFrame(record_counts)
    print("\\nRecord counts per table:")
    print(df_counts.to_string(index=False))


Database file size: 108.00 KB
\nRecord counts per table:
          Table  Record Count
      questions            18
sqlite_sequence             4
daily_questions            19
        players             8
        guesses           118
       messages           371
          roles             0
   player_roles             0


## 3. User Engagement Metrics

This section dives into player activity, including leaderboards, guess accuracy, and player growth over time.


In [3]:
if 'conn' in locals() and conn:
    # Player Leaderboard (by correct guesses)
    query = """
    SELECT p.name, COUNT(g.id) as correct_guesses
    FROM guesses g
    JOIN players p ON g.player_id = p.id
    WHERE g.is_correct = 1
    GROUP BY p.name
    ORDER BY correct_guesses DESC;
    """
    df_leaderboard = pd.read_sql_query(query, conn)
    print("--- Player Leaderboard (Correct Guesses) ---")
    print(df_leaderboard.to_string(index=False))

    # Guess Distribution (Correct vs. Incorrect)
    query = "SELECT is_correct, COUNT(id) as count FROM guesses GROUP BY is_correct;"
    df_guesses = pd.read_sql_query(query, conn)
    df_guesses['is_correct'] = df_guesses['is_correct'].apply(lambda x: 'Correct' if x == 1 else 'Incorrect')
    fig = px.pie(df_guesses, names='is_correct', values='count', title='Guess Distribution (Correct vs. Incorrect)')
    fig.show()

    # Player Growth Over Time
    query = "SELECT id, name, created_at FROM players ORDER BY created_at;"
    df_players = pd.read_sql_query(query, conn)
    df_players['created_at'] = pd.to_datetime(df_players['created_at'])
    df_players['player_count'] = range(1, len(df_players) + 1)
    fig = px.line(df_players, x='created_at', y='player_count', title='Player Growth Over Time')
    fig.update_xaxes(title_text='Date')
    fig.update_yaxes(title_text='Total Players')
    fig.show()


--- Player Leaderboard (Correct Guesses) ---
        name  correct_guesses
       Blake               16
HyruleanMyth                9
   Juniornlx                7
       Kenny                5
  Bluefox707                5
    Leilanie                3
     Kristen                3
         Amy                3


## 4. Question Usage Metrics

This section analyzes the usage of trivia questions, including the overall percentage of used questions and a breakdown by source dataset.


In [4]:
if 'conn' in locals() and conn:
    # Total questions vs. Used questions
    query_total = "SELECT COUNT(*) FROM questions;"
    total_questions = pd.read_sql_query(query_total, conn).iloc[0, 0]

    query_used = "SELECT COUNT(DISTINCT question_id) FROM daily_questions;"
    used_questions = pd.read_sql_query(query_used, conn).iloc[0, 0]

    print(f"Total questions available: {total_questions}")
    print(f"Unique questions used: {used_questions}")
    if total_questions > 0:
        print(f"Percentage of questions used: {used_questions / total_questions:.2%}")

    # Question usage by source
    query = """
    SELECT
        q.source,
        COUNT(q.id) as total_questions,
        COUNT(dq.question_id) as used_questions
    FROM questions q
    LEFT JOIN (SELECT DISTINCT question_id FROM daily_questions) dq ON q.id = dq.question_id
    GROUP BY q.source;
    """
    df_source = pd.read_sql_query(query, conn)
    df_source['usage_percentage'] = (df_source['used_questions'] / df_source['total_questions']).fillna(0)

    fig = px.bar(df_source,
                 x='source',
                 y=['used_questions', 'total_questions'],
                 title='Question Usage by Source',
                 labels={'value': 'Number of Questions', 'variable': 'Status', 'source': 'Dataset'},
                 barmode='group')
    fig.show()


Total questions available: 18
Unique questions used: 18
Percentage of questions used: 100.00%


## 5. Command and Message Analysis

This section examines the commands used by players and the overall message traffic.


In [5]:
if 'conn' in locals() and conn:
    # User Command Distribution
    query = """
    SELECT content FROM messages WHERE direction = 'incoming' AND content LIKE '/%';
    """
    df_commands = pd.read_sql_query(query, conn)
    df_commands['command'] = df_commands['content'].apply(lambda x: x.split(' ')[0])
    command_counts = df_commands['command'].value_counts().reset_index()
    command_counts.columns = ['command', 'count']

    fig = px.bar(command_counts, x='command', y='count', title='User Command Distribution')
    fig.show()

    # Message Log Analysis
    query = "SELECT status, COUNT(id) as count FROM messages GROUP BY status;"
    df_messages = pd.read_sql_query(query, conn)
    fig = px.pie(df_messages, names='status', values='count', title='Message Status Distribution')
    fig.show()


## 6. Player and Data Sanity Checks

This section includes checks for player role distribution and verifies the integrity of the daily questions data.


In [6]:
if 'conn' in locals() and conn:
    # Player Role Distribution
    query = """
    SELECT r.name, COUNT(pr.player_id) as player_count
    FROM roles r
    LEFT JOIN player_roles pr ON r.id = pr.role_id
    GROUP BY r.name;
    """
    df_roles = pd.read_sql_query(query, conn)
    fig = px.pie(df_roles, names='name', values='player_count', title='Player Role Distribution')
    fig.show()

    # Daily Question Sanity Check
    query = "SELECT sent_at FROM daily_questions ORDER BY sent_at;"
    df_daily = pd.read_sql_query(query, conn)
    df_daily['sent_at'] = pd.to_datetime(df_daily['sent_at']).dt.date
    unique_dates = df_daily['sent_at'].unique()

    if len(unique_dates) > 0:
        date_range = pd.date_range(start=unique_dates.min(), end=unique_dates.max())
        missing_dates = date_range.difference(pd.to_datetime(unique_dates))
        if not missing_dates.empty:
            print("\\n--- Missing Daily Questions ---")
            print("Warning: The following dates are missing from the 'daily_questions' table:")
            for date in missing_dates:
                print(date.strftime('%Y-%m-%d'))
        else:
            print("\\nDaily question check passed: No missing dates found.")


\n--- Missing Daily Questions ---
2025-09-16
